# Introduction to Analysing TrackMate Data

This notebook is a **gentle, generic introduction** to analysing the output of a
[TrackMate](https://imagej.net/plugins/trackmate/) tracking run.

TrackMate (a Fiji/ImageJ plugin) detects objects ("spots") in each frame of a
time-lapse and links them into "tracks". It can export two CSV tables:

- **`*_tracks.csv`**: one row per track, with summary measurements
  (duration, mean speed, displacement, confinement ratio, ...).
- **`*_spots.csv`**: one row per detection, with per-spot position, time,
  intensity and shape measurements.

By the end of this notebook you will know how to:

1. Load TrackMate's CSV exports with `pandas`.
2. Explore track-level statistics (duration, speed, displacement, confinement).
3. Work with per-spot data to reconstruct trajectories and compute
   instantaneous speeds and mean-squared displacement (MSD).

Everything here is deliberately **generic**: it only relies on the standard
columns TrackMate always writes, so it should work with any TrackMate export
regardless of your biological question.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(context="notebook", style="whitegrid")


In [ ]:
# If running in Google Colab, fetch the example data so the settings cell below
# can find it. On your own computer this cell does nothing.
try:
    import google.colab
    import urllib.request

    example_dir = Path("example_input")
    example_dir.mkdir(exist_ok=True)
    base_url = "https://raw.githubusercontent.com/FrancisCrickInstitute/TrackMateAnalysis/main/example_input"
    for filename in ("example_tracks.csv", "example_spots.csv"):
        target = example_dir / filename
        if not target.exists():
            urllib.request.urlretrieve(f"{base_url}/{filename}", target)
    print("Example data downloaded for Google Colab.")
except ImportError:
    pass



## 1. The TrackMate CSV format

TrackMate writes CSVs with **four header rows** before the data:

1. feature names (e.g. `TRACK_ID`, `TRACK_MEAN_SPEED`),
2. human-readable names,
3. short names,
4. units.

`pandas` cannot handle this automatically, so we skip the last three header rows
with `skiprows=[1, 2, 3]` and use row 0 (the feature names) as column headers.


In [ ]:
def read_trackmate_csv(path):
    """Read a TrackMate-exported CSV, keeping the feature names as headers."""
    return pd.read_csv(path, skiprows=[1, 2, 3])


## 2. Point the notebook at your data

Set `DATA_DIR` to the folder that contains your exported `*_tracks.csv` and
`*_spots.csv` files. The code below searches for them recursively, so a nested
folder structure (e.g. one subfolder per imaging position) works fine.

The settings cell also lets you optionally drop tracks you don't want: set
`MIN_TRACK_DURATION` (seconds) or `MIN_TRACK_QUALITY` to a value above 0 to
filter short or low-quality tracks out of the analysis.


In [ ]:
# ===== YOUR SETTINGS - edit these =====
DATA_DIR = Path("./example_input")  # folder with your exported CSVs

# Optional cleaning: drop tracks you don't want (0 = keep everything).
MIN_TRACK_DURATION = 0   # seconds; drop tracks shorter than this
MIN_TRACK_QUALITY = 0    # drop tracks below this mean quality (0 = keep all)
# ======================================

track_files = sorted(DATA_DIR.rglob("*_tracks.csv"))
spot_files = sorted(DATA_DIR.rglob("*_spots.csv"))

print(f"Found {len(track_files)} track file(s) and {len(spot_files)} spot file(s) in {DATA_DIR}")
if not track_files or not spot_files:
    print("No TrackMate CSV files found - check that DATA_DIR is correct.")



For simplicity we load the first of each file type. If you have many files
(e.g. one per position), you can loop over them and concatenate the results
with `pd.concat(...)`.


In [ ]:
tracks = None
spots = None

if track_files and spot_files:
    tracks = read_trackmate_csv(track_files[0])
    spots = read_trackmate_csv(spot_files[0])
    print(f"Loaded tracks from {track_files[0].name}")
    print(f"Loaded spots  from {spot_files[0].name}")

    # Apply the optional filters from the settings cell above.
    before = len(tracks)
    if "TRACK_DURATION" in tracks.columns:
        tracks = tracks[tracks["TRACK_DURATION"] >= MIN_TRACK_DURATION]
    if "TRACK_MEAN_QUALITY" in tracks.columns:
        tracks = tracks[tracks["TRACK_MEAN_QUALITY"] >= MIN_TRACK_QUALITY]
    if "TRACK_ID" in tracks.columns and "TRACK_ID" in spots.columns:
        spots = spots[spots["TRACK_ID"].isin(tracks["TRACK_ID"])]
    print(f"Kept {len(tracks)} of {before} tracks after filtering")



In [ ]:
if tracks is not None:
    tracks.head()


Key track columns (see the units in the CSV header):

| column | meaning |
|---|---|
| `TRACK_ID` | unique identifier for each track |
| `NUMBER_SPOTS` | how many detections make up the track |
| `TRACK_DURATION` | track length in time units |
| `TRACK_DISPLACEMENT` | straight-line distance between first and last spot |
| `TOTAL_DISTANCE_TRAVELED` | sum of step lengths along the track |
| `TRACK_MEAN_SPEED` | `TOTAL_DISTANCE_TRAVELED / TRACK_DURATION` |
| `CONFINEMENT_RATIO` | `TRACK_DISPLACEMENT / TOTAL_DISTANCE_TRAVELED` (0-1) |
| `LINEARITY_OF_FORWARD_PROGRESSION` | how straight the track is (1 = perfectly straight) |
| `TRACK_MEAN_QUALITY` | mean detection quality of the spots in the track |


In [ ]:
if tracks is not None and not tracks.empty:
    print(f"{len(tracks)} tracks")
    tracks[["TRACK_DURATION", "TRACK_DISPLACEMENT", "TRACK_MEAN_SPEED", "CONFINEMENT_RATIO"]].describe()


## 3. Calibration

TrackMate writes positions and times in **physical units** whenever you set the
image calibration in Fiji (pixel size and frame interval). The example data here
is already in microns (`POSITION_X`, `AREA`, ...) and seconds (`POSITION_T`).

We still record the calibration explicitly so that frame- and pixel-based
quantities can be interpreted:

- **Spatial**: `MICRONS_PER_PIXEL` - the physical size of one pixel (only needed
  for raw-pixel exports; TrackMate already applies it to `POSITION_*`, `RADIUS`,
  `AREA` and `PERIMETER`).
- **Temporal**: `SECONDS_PER_FRAME` - the interval between consecutive frames,
  derived from `POSITION_T` below.

The temporal calibration is used to turn frame-based lags into real time and to
express speeds in microns per minute, which is easier to interpret than microns
per second.


In [ ]:
MICRONS_PER_PIXEL = 0.3
SECONDS_PER_FRAME = 60.0

if spots is not None and not spots.empty and {"POSITION_T", "FRAME"} <= set(spots.columns):
    first_track = spots[spots["TRACK_ID"] == spots["TRACK_ID"].iloc[0]].sort_values("FRAME")
    interval = first_track["POSITION_T"].diff().median()
    if pd.notna(interval) and interval > 0:
        SECONDS_PER_FRAME = interval

MINUTES_PER_FRAME = SECONDS_PER_FRAME / 60

print(f"Spatial: {MICRONS_PER_PIXEL:.2f} um/pixel")
print(f"Temporal: {SECONDS_PER_FRAME:.2f} s/frame = {MINUTES_PER_FRAME:.3f} min/frame")


## 4. Track-level statistics

A good first step is to look at the distribution of track durations; this
tells you how long objects were tracked for, and is a useful quality-control
measure (very short tracks are often tracking artefacts).


### Plot any track metric

You don't need to write a new plot each time. The notebook keeps a short
"menu" of track metrics with friendly names, and two helper functions that
turn a name into a plot:

- `plot_track_metric("Mean speed")` draws a histogram of one metric.
- `scatter_track_metrics("Mean speed", "Confinement ratio")` compares two
  metrics, optionally coloured by a third with `color_metric=...`.

Run the cell below to see the full list of names you can use.



In [ ]:
# A menu of the most useful track metrics: friendly name -> (column, units).
TRACK_METRICS = {
    "Duration": ("TRACK_DURATION", "s"),
    "Displacement": ("TRACK_DISPLACEMENT", "um"),
    "Total distance": ("TOTAL_DISTANCE_TRAVELED", "um"),
    "Mean speed": ("TRACK_MEAN_SPEED", "um/s"),
    "Max speed": ("TRACK_MAX_SPEED", "um/s"),
    "Confinement ratio": ("CONFINEMENT_RATIO", ""),
    "Linearity": ("LINEARITY_OF_FORWARD_PROGRESSION", ""),
    "Mean quality": ("TRACK_MEAN_QUALITY", ""),
    "Number of spots": ("NUMBER_SPOTS", ""),
}


def _metric_label(name, unit):
    """Build an axis label like 'Mean speed (um/s)', or just the name."""
    return f"{name} ({unit})" if unit else name


def plot_track_metric(metric, bins=30):
    """Histogram of one track metric (pick the name from the list above)."""
    if metric not in TRACK_METRICS:
        print(f"Unknown metric '{metric}'. Options: {', '.join(TRACK_METRICS)}")
        return
    if tracks is None or tracks.empty:
        print("No tracks loaded - check DATA_DIR and the filters at the top.")
        return
    column, unit = TRACK_METRICS[metric]
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(tracks[column], bins=bins, ax=ax)
    ax.set_xlabel(_metric_label(metric, unit))
    ax.set_ylabel("Number of tracks")
    plt.show()


def scatter_track_metrics(x_metric, y_metric, color_metric=None, alpha=1.0):
    """Scatter plot of two track metrics, optionally coloured by a third."""
    for metric in (x_metric, y_metric):
        if metric not in TRACK_METRICS:
            print(f"Unknown metric '{metric}'. Options: {', '.join(TRACK_METRICS)}")
            return
    hue = None
    if color_metric is not None:
        if color_metric not in TRACK_METRICS:
            print(f"Unknown metric '{color_metric}'. Options: {', '.join(TRACK_METRICS)}")
            return
        hue = TRACK_METRICS[color_metric][0]
    if tracks is None or tracks.empty:
        print("No tracks loaded - check DATA_DIR and the filters at the top.")
        return
    x_column, x_unit = TRACK_METRICS[x_metric]
    y_column, y_unit = TRACK_METRICS[y_metric]
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.scatterplot(data=tracks, x=x_column, y=y_column, hue=hue, alpha=alpha, ax=ax)
    ax.set_xlabel(_metric_label(x_metric, x_unit))
    ax.set_ylabel(_metric_label(y_metric, y_unit))
    plt.show()


print("Available track metrics:")
for name in TRACK_METRICS:
    print(f"  - {name}")



In [ ]:
plot_track_metric("Duration")


Mean speed and confinement ratio are two common ways to characterise motion.
Confinement ratio ranges from 0 (back-and-forth motion) to 1 (perfectly
straight movement). Plotting it against mean speed separates objects by their
mode of motion.


In [ ]:
plot_track_metric("Mean speed")
plot_track_metric("Confinement ratio")


In [ ]:
scatter_track_metrics("Mean speed", "Confinement ratio", color_metric="Mean quality")


## 5. Reconstructing trajectories from spots

The `*_spots.csv` table contains every detection, with its `TRACK_ID`,
`POSITION_X` / `POSITION_Y` / `POSITION_Z`, `FRAME` and `POSITION_T` (time).
By plotting the spots of a few tracks we can visualise the paths the objects
took.


In [ ]:
if spots is not None:
    spots.head()


Key spot columns:

| column | meaning |
|---|---|
| `TRACK_ID` | which track this spot belongs to |
| `POSITION_X`, `POSITION_Y`, `POSITION_Z` | spot coordinates |
| `FRAME` | frame number the spot was detected in |
| `POSITION_T` | time (see units) the spot was detected at |
| `QUALITY` | detection quality score |
| `RADIUS` | estimated spot radius |

Channel intensities (e.g. `MEAN_INTENSITY_CH1`) and shape features are also
exported when available; their names depend on your detector settings.


In [ ]:
if spots is not None and not spots.empty:
    example_tracks = sorted(spots["TRACK_ID"].unique())[:20]

    fig, ax = plt.subplots(figsize=(8, 8))
    for track_id in example_tracks:
        track_spots = spots[spots["TRACK_ID"] == track_id].sort_values("FRAME")
        ax.plot(
            track_spots["POSITION_X"],
            track_spots["POSITION_Y"],
            marker="o",
            markersize=2,
            linewidth=1,
            label=f"Track {track_id}",
        )
    ax.set_xlabel("X (um)")
    ax.set_ylabel("Y (um)")
    ax.legend()
    ax.set_aspect("equal")
    plt.show()


## 6. Instantaneous speed

TrackMate's `TRACK_MEAN_SPEED` is the *average* speed over a whole track
(total distance divided by duration). We can also compute the **instantaneous**
speed between consecutive frames from the spots data, which reveals how speed
changes over time.


In [ ]:
spots_sorted = None

if spots is not None and not spots.empty:
    spots_sorted = spots.sort_values(["TRACK_ID", "FRAME"]).copy()
    by_track = spots_sorted.groupby("TRACK_ID")

    spots_sorted["DX"] = by_track["POSITION_X"].diff()
    spots_sorted["DY"] = by_track["POSITION_Y"].diff()
    spots_sorted["DT"] = by_track["POSITION_T"].diff()
    spots_sorted["INSTANT_SPEED"] = (
        np.sqrt(spots_sorted["DX"] ** 2 + spots_sorted["DY"] ** 2) / spots_sorted["DT"]
    )

    # The first spot of each track has no previous spot to compare with.
    spots_sorted["INSTANT_SPEED"] = spots_sorted["INSTANT_SPEED"].replace([np.inf, -np.inf], np.nan)

    print("Added DX, DY, DT and INSTANT_SPEED columns:")
    spots_sorted[["TRACK_ID", "FRAME", "POSITION_X", "POSITION_Y", "INSTANT_SPEED"]].head()


In [ ]:
if spots_sorted is not None and not spots_sorted.empty:
    example_track = sorted(spots_sorted["TRACK_ID"].unique())[0]
    one_track = spots_sorted[spots_sorted["TRACK_ID"] == example_track]

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(one_track["POSITION_T"] / 60, one_track["INSTANT_SPEED"] * 60, marker="o", markersize=2)
    ax.set_xlabel("Time (min)")
    ax.set_ylabel("Instantaneous speed (um/min)")
    ax.set_title(f"Track {example_track}")
    plt.show()


## 7. Mean squared displacement (MSD)

The **mean squared displacement** measures how far a particle has moved from its
starting point on average, as a function of time lag. For pure diffusion, MSD
grows linearly with lag; for directed (persistent) motion it grows faster.

We compute MSD per track, then plot a few examples. The lag axis is converted
from frames to minutes using the temporal calibration defined above.

Note: this assumes one detection per frame with no gaps. If a track
skips frames, the frame-based lag no longer equals the true elapsed time
and the MSD curve is distorted.


In [ ]:
def track_msd(x, y):
    """Mean squared displacement vs. lag (in frames) for a single track."""
    x = np.asarray(x)
    y = np.asarray(y)
    n = len(x)
    msd = []
    for lag in range(1, n):
        dx = x[lag:] - x[:-lag]
        dy = y[lag:] - y[:-lag]
        msd.append(np.mean(dx ** 2 + dy ** 2))
    return msd


In [ ]:
if spots_sorted is not None and not spots_sorted.empty:
    example_tracks = sorted(spots_sorted["TRACK_ID"].unique())[:4]

    fig, ax = plt.subplots(figsize=(8, 6))
    for track_id in example_tracks:
        track_spots = spots_sorted[spots_sorted["TRACK_ID"] == track_id].sort_values("FRAME")
        msd = track_msd(track_spots["POSITION_X"], track_spots["POSITION_Y"])
        ax.plot(np.arange(1, len(msd) + 1) * MINUTES_PER_FRAME, msd, label=f"Track {track_id}")

    ax.set_xlabel("Lag (min)")
    ax.set_ylabel("Mean squared displacement (um^2)")
    ax.legend()
    plt.show()


## 8. Morphological analysis

Beyond position and intensity, the `*_spots.csv` table often carries **shape**
features for each detection. Two of the most useful are `AREA` and
`CIRCULARITY`, but TrackMate can also export `PERIMETER`, `SOLIDITY` and
`ELLIPSE_ASPECTRATIO` (the exact set depends on your detector).

| column | meaning |
|---|---|
| `AREA` | projected area of the detection (see units) |
| `PERIMETER` | length of the detection outline |
| `CIRCULARITY` | `4 * pi * AREA / PERIMETER^2`, 0-1 (1 = perfect circle) |
| `SOLIDITY` | `AREA / convex-hull area` (1 = no concave regions) |
| `ELLIPSE_ASPECTRATIO` | long axis / short axis of the fitted ellipse |

`CIRCULARITY` and `SOLIDITY` are **size-independent** shape descriptors, so they
can be compared across objects of very different sizes.


In [ ]:
if spots is not None and not spots.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    if "AREA" in spots.columns:
        sns.histplot(spots["AREA"], bins=30, ax=axes[0])
        axes[0].set_xlabel("Area (um^2)")

    if "CIRCULARITY" in spots.columns:
        sns.histplot(spots["CIRCULARITY"], bins=30, ax=axes[1])
        axes[1].set_xlabel("Circularity (dimensionless)")

    plt.tight_layout()
    plt.show()


In [ ]:
if spots is not None and not spots.empty and {"AREA", "CIRCULARITY"} <= set(spots.columns):
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.scatterplot(
        data=spots,
        x="AREA",
        y="CIRCULARITY",
        hue="QUALITY",
        alpha=0.5,
        ax=ax,
    )
    ax.set_xlabel("Area (um^2)")
    ax.set_ylabel("Circularity (dimensionless)")
    plt.show()


## 9. How does size change over time?

A spot's `AREA` is a handy proxy for its size (or length). By plotting area
against time for a single track we can watch an object grow or shrink, and by
computing the *change* in area between consecutive frames we can see how fast
that happens.

This uses the same idea as the instantaneous-speed section: divide the
difference between consecutive frames by the time between them.



In [ ]:
if spots_sorted is not None and not spots_sorted.empty and "AREA" in spots_sorted.columns:
    by_track = spots_sorted.groupby("TRACK_ID")
    spots_sorted["DAREA_DT"] = by_track["AREA"].diff() / by_track["POSITION_T"].diff()
    print("Added DAREA_DT (rate of area change, in um^2/s).")



In [ ]:
if spots_sorted is not None and not spots_sorted.empty and "AREA" in spots_sorted.columns:
    example_tracks = sorted(spots_sorted["TRACK_ID"].unique())[:3]

    fig, ax = plt.subplots(figsize=(9, 4))
    for track_id in example_tracks:
        one_track = spots_sorted[spots_sorted["TRACK_ID"] == track_id]
        ax.plot(
            one_track["POSITION_T"] / 60,
            one_track["AREA"],
            marker="o",
            markersize=2,
            label=f"Track {track_id}",
        )
    ax.set_xlabel("Time (min)")
    ax.set_ylabel("Area (um^2)")
    ax.legend()
    plt.show()



In [ ]:
if spots_sorted is not None and not spots_sorted.empty and "DAREA_DT" in spots_sorted.columns:
    example_tracks = sorted(spots_sorted["TRACK_ID"].unique())[:3]

    fig, ax = plt.subplots(figsize=(9, 4))
    for track_id in example_tracks:
        one_track = spots_sorted[spots_sorted["TRACK_ID"] == track_id]
        ax.plot(
            one_track["POSITION_T"] / 60,
            one_track["DAREA_DT"] * 60,
            marker="o",
            markersize=2,
            label=f"Track {track_id}",
        )
    ax.set_xlabel("Time (min)")
    ax.set_ylabel("Area change rate (um^2/min)")
    ax.legend()
    plt.show()



## 10. Try it yourself

Now explore your own data. You only need to change two things:

- the **settings** at the top (`DATA_DIR`, `MIN_TRACK_DURATION`,
  `MIN_TRACK_QUALITY`), and
- the **metric names** in the `plot_track_metric(...)` and
  `scatter_track_metrics(...)` calls below.

Some ideas to get started (run a cell, then change the metric name and run it
again):

- `plot_track_metric("Displacement")` - how far tracks move overall.
- `plot_track_metric("Number of spots")` - how long tracks last, in detections.
- `scatter_track_metrics("Duration", "Total distance")` - do longer tracks
  travel further?
- Set `MIN_TRACK_DURATION = 11000` at the top, then re-run the notebook with
  Kernel -> Run All. Do the distributions change when the shortest tracks are
  dropped?
- In the size-over-time section, change `"AREA"` to `"CIRCULARITY"` or
  `"RADIUS"` to plot a different feature over time.

Per-spot shape features (area, circularity) are in the morphology section
above; size and its rate of change over time are in the size-over-time section;
speed over time is in the instantaneous-speed and MSD sections.



In [ ]:
plot_track_metric("Displacement")


In [ ]:
scatter_track_metrics("Duration", "Total distance", color_metric="Mean quality")


## Summary & next steps

You have seen how to:

- load TrackMate's spots/tracks CSVs and understand their layout,
- summarise and visualise track-level statistics,
- reconstruct trajectories and instantaneous speeds from spots,
- compute mean squared displacement per track,
- explore per-spot shape features (area, circularity),
- track how a spot's area (size) changes over time and how fast it changes.

Things you might do next:

- **Compare conditions**: concatenate tracks from multiple files and add a
  condition/position column, then group by it.
- **Filter tracks**: e.g. keep only tracks longer than a minimum duration or
  above a quality threshold before drawing conclusions.
- **Plot intensity over time**: if your spots contain channel intensities
  (e.g. `MEAN_INTENSITY_CH1`), plot them per track to relate motion to
  fluorescent reporters.
- **Use data from other sources**: rename your columns to the TrackMate names
  (`TRACK_ID`, `FRAME`, `POSITION_T`, `POSITION_X`, `POSITION_Y`) and the
  notebook will work unchanged.
- **Statistical testing**: use `scipy.stats` to test whether speed differs
  between groups.

The functions in this notebook are intentionally simple so they are easy to
adapt to your own experiment.

